In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/alpha_ml")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
EXPERIMENT_DIR = NOTEBOOK_DIR / "experiments"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

print("Project directory:", PROJECT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Experiment directory:", EXPERIMENT_DIR)

In [ ]:
%%writefile /content/drive/MyDrive/alpha_ml/.gitignore
# Large or reproducible project artifacts
data/
processed/
cache/
checkpoints/
models/
predictions/
portfolio/

# Private local configuration
config.json
.env

# Model and array files
*.pt
*.pth
*.joblib
*.pkl
*.npy
*.npz
*.parquet

# Python and notebook files
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store

# This maintenance notebook is not part of the research pipeline
notebooks/documentation_and_github.ipynb
notebooks/alpha_ml_documentation_and_github.ipynb

In [ ]:
%%writefile /content/drive/MyDrive/alpha_ml/requirements.txt
numpy
pandas
scipy
matplotlib
seaborn
scikit-learn
lightgbm
xgboost
torch
yfinance
pandas-datareader
cvxpy
joblib
pyarrow

In [ ]:
%%writefile /content/drive/MyDrive/alpha_ml/LICENSE
MIT License

Copyright (c) 2026 Yi-Heng Tsai

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:
%%writefile /content/drive/MyDrive/alpha_ml/README.md
# Alpha ML: Cross-Sectional Equity Ranking

This project develops an end-to-end machine learning pipeline for cross-sectional equity ranking. The models use historical price, volume, volatility, moving-average, and rank-based features to estimate whether each stock's future relative return will fall in the daily bottom 20%, middle 60%, or top 20%.

The primary experiment uses a 252-trading-day lookback. Alternative 60-day-lookback and directional-target versions are retained in `notebooks/experiments/`.

## Project workflow

1. Collect and clean daily equity market data.
2. Construct return, volatility, trend, liquidity, and cross-sectional rank features.
3. Create multi-horizon targets for 1, 3, 5, 10, 20, and 60 trading days.
4. Prevent look-forward leakage by requiring each target end date to remain inside its train or validation split.
5. Compare LightGBM with FCNN, CNN, GRU, Transformer, CNN+GRU, and CNN+Transformer models.
6. Select the best model for each horizon using validation mean daily cross-sectional AUC.
7. Refit the selected models on train plus validation data and generate out-of-sample probabilities.
8. Test market-neutral portfolio rules with overlapping holding periods and transaction costs.

The neural networks first encode each stock's historical sequence and then use cross-sectional attention so that predictions for one stock can incorporate information from the other stocks available on the same date.

## Main modeling result

The 252-day-lookback experiment produced out-of-sample mean daily macro AUC values of approximately 0.63 across the tested horizons. This indicates measurable cross-sectional ranking information, but predictive accuracy alone did not translate into a robust tradable strategy.

## Why the portfolio was not economically effective

The strongest directional market-neutral test retained a modest gross signal, but most of the edge disappeared after transaction costs.

| Test metric | Result |
|---|---:|
| Gross Sharpe ratio | 0.491 |
| Net Sharpe ratio at 10 bps | 0.017 |
| Net annualized return | -0.08% |
| Maximum drawdown | -7.54% |
| Mean daily turnover | 5.59% |
| Estimated break-even cost | 10.35 bps |

The main reasons are:

- classification AUC measures ranking quality, not return magnitude or portfolio Sharpe ratio;
- the predictive edge was small relative to turnover and assumed trading costs;
- dollar neutrality did not eliminate beta, sector, or other factor exposures;
- performance was unstable across market regimes and was weaker on the short side;
- overlapping targets and estimation error reduced effective out-of-sample robustness.

The negative portfolio result is kept intentionally. It demonstrates the difference between statistically detectable prediction ability and economically useful performance after implementation costs.

## Repository structure

```text
alpha_ml/
├── notebooks/                  # Main data, feature, and modeling notebooks
│   └── experiments/            # 60-day and directional research variants
├── figures/                    # Selected charts for documentation
├── results/                    # Small summary tables, when included
├── requirements.txt
├── LICENSE
└── README.md
```

Large datasets, fitted models, checkpoints, prediction files, and portfolio-level intermediate outputs are excluded from GitHub.

## Running the project

1. Install the packages in `requirements.txt`.
2. Run the data collection and feature engineering notebooks in order.
3. Run the primary 252-day-lookback model notebook with a GPU runtime.
4. Use the notebooks in `notebooks/experiments/` for alternative lookbacks and directional portfolio research.

Full neural-network training is computationally expensive. A reduced development configuration should be used first to verify the pipeline before running all horizons and model architectures.

## Limitations

- The available constituent list can introduce survivorship bias.
- The portfolio tests do not yet fully neutralize beta, sector, and common risk-factor exposures.
- Model selection is based on predictive metrics rather than a differentiable portfolio objective.
- A single train-validation-test split does not measure stability as thoroughly as walk-forward evaluation.

Future work will explore alternative targets and losses, factor neutralization, turnover-aware objectives, and walk-forward testing.

This repository is a research project and does not constitute investment advice.


## Prepare the Git repository

The following cell initializes Git only if necessary, keeps the existing notebook structure unchanged, stages the allowed files, and creates a commit only when there are changes.


In [ ]:
import subprocess


def run_git(*arguments, check=True, capture_output=False, env=None):
    return subprocess.run(
        ["git", "-C", str(PROJECT_DIR), *arguments],
        check=check,
        text=True,
        capture_output=capture_output,
        env=env,
    )


if not (PROJECT_DIR / ".git").exists():
    run_git("init", "-b", "main")

run_git("config", "user.name", "Yi-Heng Tsai")
run_git("config", "user.email", "yiheng870106@users.noreply.github.com")

github_url = "https://github.com/yiheng870106/alpha_ml.git"
remotes = run_git("remote", capture_output=True).stdout.split()
if "origin" in remotes:
    run_git("remote", "set-url", "origin", github_url)
else:
    run_git("remote", "add", "origin", github_url)

run_git("add", ".")
status = run_git("status", "--short", capture_output=True).stdout
print(status if status else "No uncommitted changes.")

if status:
    run_git("commit", "-m", "Document alpha_ml research pipeline")
else:
    print("Nothing new to commit.")


## Push to GitHub

Create a fine-grained personal access token restricted to the `alpha_ml` repository with **Contents: Read and write** permission. The token entered below is used only for this push and is not saved in the notebook or Git remote.

This cell does not use force push. If GitHub already contains unrelated commits, Git will stop instead of overwriting them.


In [ ]:
import os
import stat
from getpass import getpass

github_token = getpass("GitHub fine-grained token: ")
askpass_path = Path("/tmp/alpha_ml_git_askpass.sh")
askpass_path.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) printf '%s\\n' \"$GITHUB_USERNAME\" ;;\n"
    "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\n"
    "esac\n"
)
askpass_path.chmod(askpass_path.stat().st_mode | stat.S_IXUSR)

push_environment = os.environ.copy()
push_environment["GIT_ASKPASS"] = str(askpass_path)
push_environment["GIT_TERMINAL_PROMPT"] = "0"
push_environment["GITHUB_USERNAME"] = "yiheng870106"
push_environment["GITHUB_TOKEN"] = github_token

try:
    run_git("push", "-u", "origin", "main", env=push_environment)
    print("Pushed successfully: https://github.com/yiheng870106/alpha_ml")
finally:
    github_token = None
    push_environment.pop("GITHUB_TOKEN", None)
    askpass_path.unlink(missing_ok=True)


In [ ]:
print("Git status:")
run_git("status", "--short", "--branch")
print("Repository: https://github.com/yiheng870106/alpha_ml")
